# 서울특별시 장애인 뉴스 크롤러

- 키워드당 30건씩: 장애인 복지 서울 / 장애인 지원 서울 / 장애인 콜택시 서울
- 네이버 뉴스 검색 → 무한 스크롤, '네이버뉴스' 라벨 붙은 기사만
- **정확도 필터**: 헤드라인+본문을 합친 텍스트에 검색 키워드 구성 단어(예: 장애인/복지/서울)가 모두 포함된 기사만 저장
- 수집 항목: 헤드라인 / 본문 요약 / 언론사명 / URL

In [2]:
import time
import random
import csv
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

# (검색어, 카테고리, 헤드라인+본문에 반드시 모두 포함되어야 하는 단어들)
KEYWORDS = [
    ("장애인 복지 서울", "복지지원사업", ("장애인", "복지", "서울")),
    ("장애인 지원 서울", "복지지원사업", ("장애인", "지원", "서울")),
    ("장애인 콜택시 서울", "장애인콜택시", ("장애인", "콜택시", "서울")),
]

TARGET_COUNT = 30
MAX_SCROLL_NO_CHANGE = 8

TEXT_BLOCK_SEL = "div.m9EMVjFAOiEZp87Q"
HEADLINE_SEL = "span.sds-comps-text-type-headline1"
BODY_SEL = "span.sds-comps-text-type-body1"
PROFILE_BLOCK_SEL = "div.sds-comps-profile"
SOURCE_LINK_SEL = "div.sds-comps-profile-info-subtexts span:nth-child(4) > a"
SOURCE_LABEL_SEL = "span.sds-comps-text-weight-sm"
PRESS_NAME_SEL = "div.sds-comps-profile-info-title span > a > span.sds-comps-text-weight-sm"

CSV_FIELDS = ["category", "keyword", "headline", "body", "press", "url", "crawled_at"]


def build_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def go_to_news_tab(driver, search_query):
    driver.get(f"https://search.naver.com/search.naver?where=news&query={search_query}")
    try:
        WebDriverWait(driver, 8).until(EC.presence_of_element_located((By.CSS_SELECTOR, TEXT_BLOCK_SEL)))
        time.sleep(random.uniform(1, 1.5))
        return True
    except TimeoutException:
        print(f"[{search_query}] 검색 결과 카드가 로드되지 않음")
        return False


def parse_card(text_block, required_terms):
    """카드 하나(text_block)에서 헤드라인/본문/언론사/URL 추출.
    '네이버뉴스' 라벨이 붙어 있고, 헤드라인+본문을 합친 텍스트에 required_terms가
    (띄어쓰기 무시하고) 전부 들어있는 기사만 통과시킨다."""
    try:
        headline = text_block.find_element(By.CSS_SELECTOR, HEADLINE_SEL).text.strip()
    except NoSuchElementException:
        return None
    if not headline:
        return None

    try:
        body = text_block.find_element(By.CSS_SELECTOR, BODY_SEL).text.strip()
    except NoSuchElementException:
        body = ""

    combined = (headline + " " + body).replace(" ", "")
    if not all(term.replace(" ", "") in combined for term in required_terms):
        return None

    try:
        card = text_block.find_element(By.XPATH, "..")
        profile_block = card.find_element(By.CSS_SELECTOR, PROFILE_BLOCK_SEL)
        source_anchor = profile_block.find_element(By.CSS_SELECTOR, SOURCE_LINK_SEL)
        source_label = source_anchor.find_element(By.CSS_SELECTOR, SOURCE_LABEL_SEL).text.strip()
    except NoSuchElementException:
        return None

    if source_label != "네이버뉴스":
        return None

    url = source_anchor.get_attribute("href")
    if not url:
        return None

    try:
        press = profile_block.find_element(By.CSS_SELECTOR, PRESS_NAME_SEL).text.strip()
    except NoSuchElementException:
        press = ""

    return headline, body, press, url


def crawl_keyword(driver, search_query, category, required_terms, seen_urls, result):
    if not go_to_news_tab(driver, search_query):
        return

    collected = 0
    no_change_count = 0
    last_raw_count = 0

    while collected < TARGET_COUNT and no_change_count < MAX_SCROLL_NO_CHANGE:
        text_blocks = driver.find_elements(By.CSS_SELECTOR, TEXT_BLOCK_SEL)
        raw_count = len(text_blocks)

        for block in text_blocks:
            try:
                parsed = parse_card(block, required_terms)
            except (StaleElementReferenceException, WebDriverException):
                continue
            if parsed is None:
                continue
            headline, body, press, url = parsed
            if url in seen_urls:
                continue

            seen_urls.add(url)
            result.append({
                "category": category, "keyword": search_query,
                "headline": headline, "body": body, "press": press, "url": url,
                "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            collected += 1
            if collected >= TARGET_COUNT:
                break

        print(f"[{category}|{search_query}] 누적 수집 {collected}건 (로드된 카드 {raw_count}개)")

        # 종료 판단은 '저장된 기사 수'가 아니라 '로드된 카드 총 개수' 기준으로 한다.
        # 정확도 필터를 통과하는 기사가 우연히 안 늘어도 페이지에 더 불러올 카드가
        # 있으면 스크롤을 계속해야 하기 때문.
        if raw_count == last_raw_count:
            no_change_count += 1
        else:
            no_change_count = 0
        last_raw_count = raw_count

        if collected >= TARGET_COUNT:
            break

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(1.5, 2.5))

    print(f"[{category}|{search_query}] 수집 종료 - 총 {collected}건\n")


driver = build_driver()
seen_urls = set()
result = []

try:
    for search_query, category, required_terms in KEYWORDS:
        print(f"\n=== '{search_query}' ({category}) 수집 시작 ===")
        crawl_keyword(driver, search_query, category, required_terms, seen_urls, result)
finally:
    driver.quit()

filename = f"disability_news_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
with open(filename, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    writer.writerows(result)

print(f"\n총 {len(result)}건 저장 완료 → {filename}")


=== '장애인 복지 서울' (복지지원사업) 수집 시작 ===
[복지지원사업|장애인 복지 서울] 누적 수집 5건 (로드된 카드 10개)
[복지지원사업|장애인 복지 서울] 누적 수집 7건 (로드된 카드 20개)
[복지지원사업|장애인 복지 서울] 누적 수집 14건 (로드된 카드 30개)
[복지지원사업|장애인 복지 서울] 누적 수집 16건 (로드된 카드 40개)
[복지지원사업|장애인 복지 서울] 누적 수집 20건 (로드된 카드 50개)
[복지지원사업|장애인 복지 서울] 누적 수집 22건 (로드된 카드 60개)
[복지지원사업|장애인 복지 서울] 누적 수집 22건 (로드된 카드 70개)
[복지지원사업|장애인 복지 서울] 누적 수집 25건 (로드된 카드 80개)
[복지지원사업|장애인 복지 서울] 누적 수집 27건 (로드된 카드 90개)
[복지지원사업|장애인 복지 서울] 누적 수집 30건 (로드된 카드 100개)
[복지지원사업|장애인 복지 서울] 수집 종료 - 총 30건


=== '장애인 지원 서울' (복지지원사업) 수집 시작 ===
[복지지원사업|장애인 지원 서울] 누적 수집 2건 (로드된 카드 10개)
[복지지원사업|장애인 지원 서울] 누적 수집 3건 (로드된 카드 20개)
[복지지원사업|장애인 지원 서울] 누적 수집 4건 (로드된 카드 30개)
[복지지원사업|장애인 지원 서울] 누적 수집 6건 (로드된 카드 40개)
[복지지원사업|장애인 지원 서울] 누적 수집 6건 (로드된 카드 50개)
[복지지원사업|장애인 지원 서울] 누적 수집 7건 (로드된 카드 60개)
[복지지원사업|장애인 지원 서울] 누적 수집 7건 (로드된 카드 70개)
[복지지원사업|장애인 지원 서울] 누적 수집 8건 (로드된 카드 80개)
[복지지원사업|장애인 지원 서울] 누적 수집 12건 (로드된 카드 90개)
[복지지원사업|장애인 지원 서울] 누적 수집 18건 (로드된 카드 100개)
[복지지원사업|장애인 지원 서울] 누적 수집 20건 (로드된 카드 110개)
[복지지원사업|장애인 지원 서울] 